In [1]:
# VERBA — Multilingual Translation Prototype

### Connecting Languages, Connecting People

"VERBA is an academic multilingual translation prototype..."

'VERBA is an academic multilingual translation prototype...'

In [2]:

# Install the libraries required by the translation model

%pip install -q "transformers<5" sentencepiece accelerate gradio

print("Dependencies installed successfully.")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
Dependencies installed successfully.


In [3]:

from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [4]:

# Import required libraries

import os
import sys
import time
import csv
import statistics

import torch
import transformers
import sentencepiece
import accelerate
import gradio as gr

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Select available hardware
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Display environment information
print("VERBA Environment")
print("-" * 35)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("SentencePiece:", sentencepiece.__version__)
print("Accelerate:", accelerate.__version__)
print("Gradio:", gr.__version__)
print("Device:", DEVICE)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    print("GPU not detected. Running on CPU.")


VERBA Environment
-----------------------------------
Python: 3.13.16
PyTorch: 2.14.1+cu130
Transformers: 4.57.6
SentencePiece: 0.2.2
Accelerate: 1.15.0
Gradio: 6.17.3
Device: cuda
GPU: Tesla T4
VRAM: 14.56 GB


In [5]:
# Load the pretrained NLLB-200 translation model

MODEL_NAME = "facebook/nllb-200-distilled-600M"
MODEL_PATH = "/content/drive/MyDrive/VERBA/models/nllb-200-distilled-600M"

MODEL_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print("Loading VERBA translation model from Hugging Face...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=MODEL_DTYPE
)

model = model.to(DEVICE)
model.eval()

# Save the complete checkpoint to Google Drive
model.save_pretrained(MODEL_PATH, safe_serialization=True)
tokenizer.save_pretrained(MODEL_PATH)

print("\nVERBA model loaded and saved successfully!")
print("Parameters:", round(sum(p.numel() for p in model.parameters()) / 1_000_000, 2), "million")
print("Device:", DEVICE)
print("Precision:", MODEL_DTYPE)
print("Checkpoint:", MODEL_PATH)

Loading VERBA translation model from Hugging Face...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

... (3 outputs left out)

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(



VERBA model loaded and saved successfully!
Parameters: 615.07 million
Device: cuda
Precision: torch.float16
Checkpoint: /content/drive/MyDrive/VERBA/models/nllb-200-distilled-600M


In [6]:
# Configure supported languages


# VERBA — Expanded Language Configuration

LANGUAGES = {
    # Indian Languages
    "English": "eng_Latn",
    "Hindi": "hin_Deva",
    "Marathi": "mar_Deva",
    "Gujarati": "guj_Gujr",
    "Bengali": "ben_Beng",
    "Tamil": "tam_Taml",
    "Telugu": "tel_Telu",
    "Punjabi": "pan_Guru",
    "Kannada": "kan_Knda",
    "Malayalam": "mal_Mlym",
    "Urdu": "urd_Arab",

    # Asian Languages
    "Japanese": "jpn_Jpan",
    "Chinese": "zho_Hans",
    "Korean": "kor_Hang",
    "Thai": "tha_Thai",
    "Indonesian": "ind_Latn",

    # European Languages
    "Spanish": "spa_Latn",
    "French": "fra_Latn",
    "German": "deu_Latn",
    "Italian": "ita_Latn",
    "Portuguese": "por_Latn",
    "Russian": "rus_Cyrl"
}

print("VERBA supports", len(LANGUAGES), "languages.")



# VERBA — Updated Translation Function

def translate(text, source_language, target_language):

    if not text or not text.strip():
        raise ValueError("Please enter some text.")

    if source_language not in LANGUAGES:
        raise ValueError("Invalid source language.")

    if target_language not in LANGUAGES:
        raise ValueError("Invalid target language.")

    if source_language == target_language:
        return text

    tokenizer.src_lang = LANGUAGES[source_language]

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(DEVICE)

    target_id = tokenizer.convert_tokens_to_ids(
        LANGUAGES[target_language]
    )

    if target_id == tokenizer.unk_token_id:
        raise ValueError("Unsupported language code.")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            forced_bos_token_id=target_id,
            max_new_tokens=128,
            num_beams=4
        )

    translated_text = tokenizer.batch_decode(
        output,
        skip_special_tokens=True
    )[0]

    return translated_text


print("VERBA Translation Engine Ready!")



# VERBA — Language Verification

print("VERBA Language Verification")
print("-" * 40)

for language, code in LANGUAGES.items():

    token_id = tokenizer.convert_tokens_to_ids(code)

    if token_id == tokenizer.unk_token_id:
        print(language, "❌ Not supported")
    else:
        print(language, "✅ Available")

print("\nLanguage verification completed!")



VERBA supports 22 languages.
VERBA Translation Engine Ready!
VERBA Language Verification
----------------------------------------
English ✅ Available
Hindi ✅ Available
Marathi ✅ Available
Gujarati ✅ Available
Bengali ✅ Available
Tamil ✅ Available
Telugu ✅ Available
Punjabi ✅ Available
Kannada ✅ Available
Malayalam ✅ Available
Urdu ✅ Available
Japanese ✅ Available
Chinese ✅ Available
Korean ✅ Available
Thai ✅ Available
Indonesian ✅ Available
Spanish ✅ Available
French ✅ Available
German ✅ Available
Italian ✅ Available
Portuguese ✅ Available
Russian ✅ Available

Language verification completed!


In [7]:

# VERBA — Multilingual Translation Demonstration

sample_text = "Welcome to Palghar. We are building a better future together."

target_languages = [
    "Hindi",
    "Marathi",
    "Gujarati",
    "Bengali",
    "Tamil",
    "Telugu",
    "Spanish",
    "French",
    "Japanese",
    "German"
]

print("VERBA Translation Demonstration")
print("-" * 50)
print("Source English:", sample_text)

for language in target_languages:

    result = translate(
        sample_text,
        "English",
        language
    )

    print(f"\n{language}:")
    print(result)


VERBA Translation Demonstration
--------------------------------------------------
Source English: Welcome to Palghar. We are building a better future together.

Hindi:
पालघर में आपका स्वागत है, हम एक साथ एक बेहतर भविष्य का निर्माण कर रहे हैं।

Marathi:
पालघरमध्ये आपले स्वागत आहे. आम्ही एकत्रितपणे एक चांगले भविष्य घडवत आहोत.

Gujarati:
પાલ્ઘરમાં આપનું સ્વાગત છે. અમે સાથે મળીને વધુ સારા ભવિષ્યનું નિર્માણ કરી રહ્યા છીએ.

Bengali:
পলগারে স্বাগতম, আমরা একসাথে আরও ভালো ভবিষ্যৎ গড়ে তুলছি।

Tamil:
பால்காரில் வரவேற்கிறோம். நாம் ஒரு சிறந்த எதிர்காலத்தை ஒன்றாக உருவாக்குகிறோம்.

Telugu:
పల్ఘర్ కు స్వాగతం. మనం కలిసి మెరుగైన భవిష్యత్తును నిర్మిస్తున్నాం.

Spanish:
Bienvenidos a Palghar, estamos construyendo un futuro mejor juntos.

French:
Bienvenue à Palghar, nous construisons ensemble un avenir meilleur.

Japanese:
パルガーへようこそ 一緒により良い未来を築こう

German:
Willkommen in Palghar, wir bauen eine bessere Zukunft zusammen.


In [8]:
# VERBA — Translation Performance Benchmark

evaluation_sentences = [
    "Welcome to Palghar.",
    "How are you today?",
    "Please bring your identity card.",
    "The hospital is located near the main road.",
    "We are building a better future together.",
    "Clean water is important for every community.",
    "The bus will arrive in ten minutes.",
    "Education creates new opportunities.",
    "Please contact us for more information.",
    "Thank you for your cooperation."
]

target_languages = ["Hindi", "Marathi", "Spanish", "Japanese"]

benchmark_results = []

# Warm-up run (not included in measurements)
translate("Welcome to Palghar.", "English", "Hindi")

if DEVICE == "cuda":
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

print("Running VERBA performance benchmark...")
print("Total translations:", len(evaluation_sentences) * len(target_languages))

for sentence in evaluation_sentences:
    for language in target_languages:

        if DEVICE == "cuda":
            torch.cuda.synchronize()

        start_time = time.perf_counter()

        translated_text = translate(
            sentence,
            "English",
            language
        )

        if DEVICE == "cuda":
            torch.cuda.synchronize()

        latency = time.perf_counter() - start_time

        benchmark_results.append({
            "source_text": sentence,
            "target_language": language,
            "translated_text": translated_text,
            "latency_seconds": round(latency, 4)
        })

# Calculate performance statistics
latencies = [
    result["latency_seconds"]
    for result in benchmark_results
]

# Save benchmark results
report_path = "/content/drive/MyDrive/VERBA/reports"

os.makedirs(report_path, exist_ok=True)

csv_path = os.path.join(
    report_path,
    "performance_benchmark.csv"
)

with open(csv_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=benchmark_results[0].keys()
    )
    writer.writeheader()
    writer.writerows(benchmark_results)

# Display summary
print("\nVERBA Performance Summary")
print("-" * 40)
print("Total translations:", len(latencies))
print("Average latency:", round(statistics.mean(latencies), 3), "seconds")
print("Median latency:", round(statistics.median(latencies), 3), "seconds")
print("Fastest translation:", round(min(latencies), 3), "seconds")
print("Slowest translation:", round(max(latencies), 3), "seconds")

for language in target_languages:
    language_times = [
        result["latency_seconds"]
        for result in benchmark_results
        if result["target_language"] == language
    ]

    print(
        f"{language}:",
        round(statistics.mean(language_times), 3),
        "seconds average"
    )

if DEVICE == "cuda":
    peak_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)
    print("Peak GPU memory:", round(peak_memory, 3), "GB")

print("\nBenchmark report saved:", csv_path)

Running VERBA performance benchmark...
Total translations: 40

VERBA Performance Summary
----------------------------------------
Total translations: 40
Average latency: 0.195 seconds
Median latency: 0.187 seconds
Fastest translation: 0.109 seconds
Slowest translation: 0.308 seconds
Hindi: 0.189 seconds average
Marathi: 0.19 seconds average
Spanish: 0.206 seconds average
Japanese: 0.194 seconds average
Peak GPU memory: 1.176 GB

Benchmark report saved: /content/drive/MyDrive/VERBA/reports/performance_benchmark.csv


In [9]:

# ============================================
# VERBA — MODERN MULTILINGUAL TRANSLATOR
# ============================================

import gradio as gr

# Translation function
def run_translation(text, source_language, target_language):

    if not text or not text.strip():
        return "Please enter text to translate."

    try:
        return translate(text, source_language, target_language)

    except Exception as error:
        return f"Translation Error: {str(error)}"


# Swap languages
def swap_languages(source, target):
    return target, source


# Clear text fields
def clear_fields():
    return "", ""


# Modern interface styling

custom_css = """
.gradio-container {
    background: linear-gradient(135deg, #10172e, #20204a, #301b52);
}

/* Header */
#main-title {
    text-align: center;
    padding: 30px;
    border-radius: 18px;
    background: linear-gradient(90deg, #292e63, #663399);
}

#main-title h1 {
    font-size: 45px;
    color: #ffffff !important;
}

#main-title p {
    color: #e1dcff !important;
}

/* Make all Markdown text visible */
.prose,
.prose p,
.prose h1,
.prose h2,
.prose h3,
.prose h4,
.prose li,
.prose strong,
.prose span {
    color: #ffffff !important;
}

/* Input labels */
label {
    color: #ffffff !important;
}

/* Buttons */
.gr-button-primary {
    background: linear-gradient(90deg, #6255e7, #9b59d0) !important;
    color: white !important;
    border: none !important;
}

/* Footer text */
footer {
    color: #ffffff !important;
}
"""


# Create VERBA application
with gr.Blocks(title="VERBA - AI Translator") as demo:

    # Header
    gr.HTML("""
    <div id="main-title">
        <h1>🌐 VERBA</h1>
        <p>Connecting Languages, Connecting People</p>
        <p>AI-Powered Multilingual Translation Platform</p>
    </div>
    """)

    gr.Markdown(
        "## ✨ Translate your thoughts into another language"
    )

    # Language selection
    with gr.Row():

        source_language = gr.Dropdown(
            choices=list(LANGUAGES.keys()),
            value="English",
            label="🌍 Translate From"
        )

        swap_button = gr.Button("⇄ Swap Languages")

        target_language = gr.Dropdown(
            choices=list(LANGUAGES.keys()),
            value="Marathi",
            label="🎯 Translate To"
        )

    # Input and output boxes
    with gr.Row():

        input_text = gr.Textbox(
            label="📝 Enter Your Text",
            placeholder="Type something here...",
            lines=8
        )

        output_text = gr.Textbox(
            label="🌐 Translation Result",
            placeholder="Your translation appears here...",
            lines=8,
            interactive=False
        )

    # Action buttons
    with gr.Row():

        translate_button = gr.Button(
            "✨ Translate Now",
            variant="primary"
        )

        clear_button = gr.Button("🗑 Clear")

    # Example sentences
    gr.Markdown("## 💡 Try These Examples")

    gr.Examples(
        examples=[
            ["Hello, how are you?", "English", "Hindi"],
            ["Welcome to Palghar.", "English", "Marathi"],
            ["Education is the key to success.", "English", "Gujarati"],
            ["Thank you for your help.", "English", "Spanish"],
            ["Good morning.", "English", "Japanese"],
            ["Technology connects the world.", "English", "French"],
            ["Knowledge is power.", "English", "German"]
        ],
        inputs=[
            input_text,
            source_language,
            target_language
        ]
    )

    # About section
    gr.Markdown("""
    ---

    ## 🚀 About VERBA

    VERBA uses Meta AI's pretrained
    **NLLB-200 Distilled 600M** translation model.

    ### Features

    - 22 language options
    - AI-powered text translation
    - Language swapping
    - Easy-to-use interface
    - Multiple language support

    ---

    ⚠️ Academic translation prototype.
    Please verify translations before important use.

    <center>
    Built with Python, PyTorch and Gradio
    | VERBA 2.0
    </center>
    """)

    # Connect buttons
    translate_button.click(
        fn=run_translation,
        inputs=[
            input_text,
            source_language,
            target_language
        ],
        outputs=output_text
    )

    swap_button.click(
        fn=swap_languages,
        inputs=[
            source_language,
            target_language
        ],
        outputs=[
            source_language,
            target_language
        ]
    )

    clear_button.click(
        fn=clear_fields,
        inputs=[],
        outputs=[
            input_text,
            output_text
        ]
    )


# Launch VERBA with custom theme and CSS
demo.launch(
    share=True,
    theme=gr.themes.Soft(),
    css=custom_css
)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://aed77f243e8846f471.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:
# VERBA 2.0 — Step 1: Install RAG libraries
# (pins keep the already-working transformers / huggingface-hub versions unchanged)

%pip install -q langchain-core langchain-chroma langchain-text-splitters chromadb pypdf sentence-transformers "transformers<5" "huggingface-hub<1.0"

print("RAG dependencies installed successfully.")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
t

In [11]:
# VERBA 2.0 — Step 2: Imports and configuration

import os
import re
import csv
import shutil
import unicodedata

import chromadb
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Folders (Google Drive is already mounted by the cell above)
VERBA_ROOT = "/content/drive/MyDrive/VERBA"
RAG_DIR = os.path.join(VERBA_ROOT, "rag_data")                 # editable CSV / TXT / PDF files
KB_DIR = os.path.join(RAG_DIR, "knowledge_base")               # optional reference documents
DB_PRIMARY = os.path.join(VERBA_ROOT, "vector_db")             # persistent ChromaDB (Drive)
DB_FALLBACK = "/content/VERBA_vector_db"                       # used only if Drive DB cannot open

GLOSSARY_CSV = os.path.join(RAG_DIR, "glossary.csv")
TM_CSV = os.path.join(RAG_DIR, "translation_memory.csv")

# Retrieval settings (same values as the project slides)
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
TOP_K = 3                  # top-3 cosine search
TM_THRESHOLD = 0.97        # minimum cosine similarity to reuse a stored translation
DOC_MIN_SIM = 0.30         # minimum similarity to show a reference-document chunk
MAX_QUERIES = 40

for folder in (RAG_DIR, KB_DIR):
    os.makedirs(folder, exist_ok=True)

print("RAG folder      :", RAG_DIR)
print("Vector DB (main):", DB_PRIMARY)


RAG folder      : /content/drive/MyDrive/VERBA/rag_data
Vector DB (main): /content/drive/MyDrive/VERBA/vector_db


In [12]:
# VERBA 2.0 — Step 3: Embedding model (LangChain-compatible wrapper)

from sentence_transformers import SentenceTransformer


class MiniLMEmbeddings(Embeddings):
    """all-MiniLM-L6-v2 embeddings for LangChain (CPU, normalized -> cosine)."""

    def __init__(self, model_name=EMBED_MODEL, device="cpu"):
        self.model = SentenceTransformer(model_name, device=device)

    def embed_documents(self, texts):
        vectors = self.model.encode(
            list(texts),
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False
        )
        return vectors.tolist()

    def embed_query(self, text):
        return self.embed_documents([text])[0]


embeddings = MiniLMEmbeddings()

print("Embedding model ready:", EMBED_MODEL)
print("Embedding dimension  :", len(embeddings.embed_query("VERBA")))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model ready: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension  : 384


In [13]:
# VERBA 2.0 — Step 4: Knowledge-base files (glossary, translation memory, reference docs)
# Files are created only if they do not already exist, so your edits are never overwritten.

GLOSSARY_FIELDS = ["source_language", "term", "target_language", "translation"]
TM_FIELDS = ["source_language", "source_text", "target_language", "translation"]

# Seed glossary: approved spellings of local names. Please verify with native speakers.
_seed_glossary = [
    ("Palghar", {"Hindi": "पालघर", "Marathi": "पालघर", "Gujarati": "પાલઘર", "Bengali": "পালঘর",
                 "Tamil": "பால்கர்", "Telugu": "పాల్ఘర్"}),
    ("Mumbai", {"Hindi": "मुंबई", "Marathi": "मुंबई", "Gujarati": "મુંબઈ", "Bengali": "মুম্বাই",
                "Tamil": "மும்பை", "Telugu": "ముంబై"}),
    ("Maharashtra", {"Hindi": "महाराष्ट्र", "Marathi": "महाराष्ट्र", "Gujarati": "મહારાષ્ટ્ર",
                     "Bengali": "মহারাষ্ট্র", "Tamil": "மகாராஷ்டிரா", "Telugu": "మహారాష్ట్ర"}),
]


def _write_csv(path, fields, rows):
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def _read_csv(path):
    with open(path, "r", newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))


def _append_csv(path, fields, row):
    new_file = not os.path.exists(path)
    with open(path, "a", newline="", encoding="utf-8-sig" if new_file else "utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        if new_file:
            writer.writeheader()
        writer.writerow(row)


if not os.path.exists(GLOSSARY_CSV):
    rows = []
    for term, translations in _seed_glossary:
        for language, translated in translations.items():
            rows.append({"source_language": "English", "term": term,
                         "target_language": language, "translation": translated})
    _write_csv(GLOSSARY_CSV, GLOSSARY_FIELDS, rows)
    print("Created glossary.csv with", len(rows), "starter entries.")

if not os.path.exists(TM_CSV):
    # Starter entry taken from VERBA's own demo output (cell above). Verify before relying on it.
    _write_csv(TM_CSV, TM_FIELDS, [{
        "source_language": "English",
        "source_text": "Welcome to Palghar. We are building a better future together.",
        "target_language": "Hindi",
        "translation": "पालघर में आपका स्वागत है, हम एक साथ एक बेहतर भविष्य का निर्माण कर रहे हैं।"
    }])
    print("Created translation_memory.csv with 1 starter entry.")

if not any(name.lower().endswith((".txt", ".pdf")) for name in os.listdir(KB_DIR)):
    with open(os.path.join(KB_DIR, "verba_terminology_guide.txt"), "w", encoding="utf-8") as f:
        f.write(
            "VERBA Terminology Guide\n\n"
            "Place names and local terms must keep their approved spelling in every target language. "
            "Names such as Palghar, Mumbai and Maharashtra are stored in the VERBA glossary with an approved "
            "translation for each supported language.\n\n"
            "Translation memory stores sentences whose translations were verified by a reviewer. "
            "When a new input is almost identical to a verified sentence, the stored translation is reused.\n\n"
            "New glossary entries can be added at any time by editing glossary.csv or by using the "
            "glossary form in the VERBA interface. No model retraining is required."
        )
    print("Created sample reference document in knowledge_base/.")

print("Glossary rows        :", len(_read_csv(GLOSSARY_CSV)))
print("Translation memory   :", len(_read_csv(TM_CSV)))
print("Reference documents  :", [n for n in os.listdir(KB_DIR) if n.lower().endswith((".txt", ".pdf"))])


Glossary rows        : 18
Translation memory   : 1
Reference documents  : ['verba_terminology_guide.txt']


In [14]:
# VERBA 2.0 — Step 5: Build the ChromaDB vector stores (LangChain Chroma)

COSINE = {"hnsw:space": "cosine"}


def _pair_filter(source_language, target_language):
    return {"$and": [
        {"source_language": {"$eq": source_language}},
        {"target_language": {"$eq": target_language}}
    ]}


def _open_store(client, name):
    return Chroma(
        client=client,
        collection_name=name,
        embedding_function=embeddings,
        collection_metadata=COSINE
    )


def _reset_collection(client, name):
    try:
        client.delete_collection(name)
    except Exception:
        pass


def _glossary_docs():
    docs, ids = {}, {}
    for row in _read_csv(GLOSSARY_CSV):
        src = (row.get("source_language") or "").strip()
        term = (row.get("term") or "").strip()
        tgt = (row.get("target_language") or "").strip()
        out = (row.get("translation") or "").strip()
        if not (src and term and tgt and out):
            continue
        if src not in LANGUAGES or tgt not in LANGUAGES:
            continue
        doc_id = f"gl::{src}::{tgt}::{term.casefold()}"
        docs[doc_id] = Document(
            page_content=term,
            metadata={"kind": "glossary", "source_language": src,
                      "target_language": tgt, "term": term, "translation": out}
        )
    return docs


def _tm_docs():
    docs = {}
    for row in _read_csv(TM_CSV):
        src = (row.get("source_language") or "").strip()
        text = (row.get("source_text") or "").strip()
        tgt = (row.get("target_language") or "").strip()
        out = (row.get("translation") or "").strip()
        if not (src and text and tgt and out):
            continue
        if src not in LANGUAGES or tgt not in LANGUAGES:
            continue
        doc_id = f"tm::{src}::{tgt}::{' '.join(text.casefold().split())}"
        docs[doc_id] = Document(
            page_content=text,
            metadata={"kind": "translation_memory", "source_language": src,
                      "target_language": tgt, "translation": out}
        )
    return docs


def _reference_chunks():
    splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    documents = []
    for name in sorted(os.listdir(KB_DIR)):
        path = os.path.join(KB_DIR, name)
        lower = name.lower()
        try:
            if lower.endswith(".txt"):
                with open(path, "r", encoding="utf-8") as f:
                    documents.append(Document(page_content=f.read(), metadata={"source": name}))
            elif lower.endswith(".pdf"):
                from pypdf import PdfReader
                for page_no, page in enumerate(PdfReader(path).pages, start=1):
                    page_text = page.extract_text() or ""
                    if page_text.strip():
                        documents.append(Document(page_content=page_text,
                                                  metadata={"source": name, "page": page_no}))
        except Exception as error:
            print("Skipped", name, "->", error)
    return splitter.split_documents(documents)


def build_vector_stores(db_path):
    os.makedirs(db_path, exist_ok=True)
    client = chromadb.PersistentClient(path=db_path)

    # Glossary and reference docs: rebuilt from the files on every run (files are the source of truth)
    _reset_collection(client, "verba_glossary")
    _reset_collection(client, "verba_reference_docs")
    glossary = _open_store(client, "verba_glossary")
    reference = _open_store(client, "verba_reference_docs")
    # Translation memory: rebuilt too, so the CSV and DB always match
    _reset_collection(client, "verba_translation_memory")
    memory = _open_store(client, "verba_translation_memory")

    g = _glossary_docs()
    if g:
        glossary.add_documents(list(g.values()), ids=list(g.keys()))

    t = _tm_docs()
    if t:
        memory.add_documents(list(t.values()), ids=list(t.keys()))

    chunks = _reference_chunks()
    if chunks:
        chunk_ids = [f"doc::{c.metadata.get('source')}::{c.metadata.get('page', 0)}::{i}"
                     for i, c in enumerate(chunks)]
        reference.add_documents(chunks, ids=chunk_ids)

    return glossary, memory, reference, len(g), len(t), len(chunks)


RAG_READY = False
try:
    try:
        DB_PATH = DB_PRIMARY
        glossary_store, tm_store, docs_store, n_gloss, n_tm, n_chunks = build_vector_stores(DB_PATH)
    except Exception as drive_error:
        print("Drive vector DB could not be opened (", drive_error, ") -> using local fallback.")
        DB_PATH = DB_FALLBACK
        glossary_store, tm_store, docs_store, n_gloss, n_tm, n_chunks = build_vector_stores(DB_PATH)
    RAG_READY = True
except Exception as rag_error:
    print("RAG setup failed:", rag_error)
    print("VERBA will keep working with the standard NLLB translation.")

if RAG_READY:
    print("VERBA vector database ready")
    print("-" * 40)
    print("Location           :", DB_PATH)
    print("Glossary entries   :", n_gloss)
    print("Translation memory :", n_tm)
    print("Reference chunks   :", n_chunks, f"(size {CHUNK_SIZE}, overlap {CHUNK_OVERLAP})")


VERBA vector database ready
----------------------------------------
Location           : /content/drive/MyDrive/VERBA/vector_db
Glossary entries   : 18
Translation memory : 1
Reference chunks   : 2 (size 500, overlap 50)


In [15]:
# VERBA 2.0 — Step 6: RAG translation pipeline (retrieve -> protect / reuse -> NLLB translate)

# Placeholder formats used to protect verified terms during NLLB generation
PLACEHOLDER_TEMPLATES = ["[{n}]", "({n})", "#{n}#"]
_OPEN = r"[\[\(\{<【（#]"
_CLOSE = r"[\]\)\}>】）#]"
PLACEHOLDER_RE = re.compile(_OPEN + r"\s*(\d{1,2})\s*" + _CLOSE)


def _norm_digits(value):
    return int("".join(str(unicodedata.digit(ch)) for ch in value))


def _numbers_in(text):
    return [_norm_digits(n) for n in re.findall(r"\d+", text)]


def _query_texts(text):
    words = re.findall(r"[^\W\d_]+", text)
    queries = [text.strip()]
    for w in words:
        if len(w) >= 3:
            queries.append(w)
    for a, b in zip(words, words[1:]):
        queries.append(a + " " + b)
    unique = []
    for q in queries:
        if q and q.casefold() not in [u.casefold() for u in unique]:
            unique.append(q)
    return unique[:MAX_QUERIES]


def _term_pattern(term):
    return re.compile(r"(?<!\w)" + re.escape(term) + r"(?!\w)", re.IGNORECASE)


def retrieve_glossary(text, source_language, target_language, k=TOP_K):
    """Vector search (LangChain retriever) -> glossary entries whose term really appears in the text."""
    retriever = glossary_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k, "filter": _pair_filter(source_language, target_language)}
    )
    candidates = {}
    for query in _query_texts(text):
        for doc in retriever.invoke(query):
            candidates[doc.metadata["term"].casefold()] = doc.metadata
    matched = [m for m in candidates.values() if _term_pattern(m["term"]).search(text)]
    matched.sort(key=lambda m: len(m["term"]), reverse=True)
    return matched


def retrieve_top_glossary(text, source_language, target_language, k=TOP_K):
    results = glossary_store.similarity_search_with_score(
        text, k=k, filter=_pair_filter(source_language, target_language)
    )
    return [(doc.metadata["term"], doc.metadata["translation"], round(1 - dist, 3))
            for doc, dist in results]


def lookup_translation_memory(text, source_language, target_language):
    results = tm_store.similarity_search_with_score(
        text, k=1, filter=_pair_filter(source_language, target_language)
    )
    if not results:
        return None
    doc, dist = results[0]
    similarity = 1 - dist
    if similarity >= TM_THRESHOLD and _numbers_in(text) == _numbers_in(doc.page_content):
        return doc.metadata["translation"], round(similarity, 3)
    return None


def retrieve_reference_chunks(text, k=TOP_K):
    if docs_store is None:
        return []
    results = docs_store.similarity_search_with_score(text, k=k)
    return [(doc.metadata.get("source", "?"), round(1 - dist, 3), doc.page_content.strip().replace("\n", " "))
            for doc, dist in results if (1 - dist) >= DOC_MIN_SIM]


def _translate_protected(text, source_language, target_language, matches):
    """Replace verified terms with placeholders, translate with NLLB, put approved terms back.
    Returns the translation, or None if NLLB did not keep every placeholder."""
    if PLACEHOLDER_RE.search(text):
        return None
    for template in PLACEHOLDER_TEMPLATES:
        protected = text
        numbered = {}
        for n, match in enumerate(matches, start=1):
            token = template.format(n=n)
            protected, count = _term_pattern(match["term"]).subn(lambda _m, t=token: t, protected)
            if count:
                numbered[n] = match
        if not numbered:
            return None
        output = translate(protected, source_language, target_language)
        found = [_norm_digits(m.group(1)) for m in PLACEHOLDER_RE.finditer(output)]
        if all(n in found for n in numbered):
            restored = PLACEHOLDER_RE.sub(
                lambda m: numbered[_norm_digits(m.group(1))]["translation"]
                if _norm_digits(m.group(1)) in numbered else m.group(0),
                output
            )
            return restored
    return None


def translate_with_rag(text, source_language, target_language, use_rag=True):
    """RAG-enhanced translation. Always falls back to the normal translate() on any problem."""
    info = {"translation": "", "method": "NLLB (standard)", "glossary_applied": [],
            "glossary_top": [], "tm_hit": None, "references": [], "notes": []}

    # Validation, same-language case and errors come from the original translate()
    if (not use_rag or not RAG_READY or not text or not text.strip()
            or source_language == target_language
            or source_language not in LANGUAGES or target_language not in LANGUAGES):
        info["translation"] = translate(text, source_language, target_language)
        if not use_rag:
            info["notes"].append("RAG is switched off.")
        return info

    clean = text.strip()

    # 1) Translation memory: reuse a verified translation on a near-exact match
    try:
        hit = lookup_translation_memory(clean, source_language, target_language)
        if hit:
            info["translation"], similarity = hit
            info["tm_hit"] = similarity
            info["method"] = "Translation Memory (verified)"
            return info
    except Exception as error:
        info["notes"].append(f"Translation memory skipped: {error}")

    # 2) Glossary: retrieve verified terms and protect them during translation
    matches = []
    try:
        matches = retrieve_glossary(clean, source_language, target_language)
        info["glossary_top"] = retrieve_top_glossary(clean, source_language, target_language)
    except Exception as error:
        info["notes"].append(f"Glossary lookup skipped: {error}")

    # 3) Reference documents (shown to the user as supporting references)
    try:
        info["references"] = retrieve_reference_chunks(clean)
    except Exception as error:
        info["notes"].append(f"Reference lookup skipped: {error}")

    # 4) NLLB translation
    result = None
    if matches:
        try:
            result = _translate_protected(clean, source_language, target_language, matches)
        except Exception as error:
            info["notes"].append(f"Term protection skipped: {error}")
        if result is not None:
            info["translation"] = result
            info["method"] = "NLLB + Glossary RAG"
            info["glossary_applied"] = [(m["term"], m["translation"]) for m in matches]
            return info
        info["notes"].append("Verified term could not be protected; standard NLLB output used.")

    info["translation"] = translate(text, source_language, target_language)
    return info


def format_rag_report(info):
    lines = [f"**Method:** {info['method']}"]
    if info["tm_hit"] is not None:
        lines.append(f"- Translation memory match (cosine {info['tm_hit']})")
    for term, translated in info["glossary_applied"]:
        lines.append(f"- Glossary applied: **{term}** → **{translated}**")
    if info["glossary_top"]:
        lines.append("")
        lines.append("**Top-3 glossary matches (cosine similarity):**")
        for term, translated, similarity in info["glossary_top"]:
            lines.append(f"- {term} → {translated} ({similarity})")
    if info["references"]:
        lines.append("")
        lines.append("**Reference chunks:**")
        for source, similarity, snippet in info["references"]:
            lines.append(f"- {source} ({similarity}): {snippet[:140]}...")
    for note in info["notes"]:
        lines.append(f"- _{note}_")
    return "\n".join(lines)


print("VERBA RAG pipeline ready." if RAG_READY else "RAG not available - standard translation only.")


VERBA RAG pipeline ready.


In [16]:
# VERBA 2.0 — Step 7: Update the vector database (no retraining needed)

def add_glossary_term(term, target_language, translation, source_language="English"):
    term, translation = (term or "").strip(), (translation or "").strip()
    if not term or not translation:
        return "Please enter both the term and its approved translation."
    if target_language not in LANGUAGES or source_language not in LANGUAGES:
        return "Invalid language."
    if not RAG_READY:
        return "RAG is not available."
    doc_id = f"gl::{source_language}::{target_language}::{term.casefold()}"
    glossary_store.add_documents(
        [Document(page_content=term, metadata={"kind": "glossary", "source_language": source_language,
                                               "target_language": target_language, "term": term,
                                               "translation": translation})],
        ids=[doc_id]
    )
    _append_csv(GLOSSARY_CSV, GLOSSARY_FIELDS, {"source_language": source_language, "term": term,
                                                "target_language": target_language,
                                                "translation": translation})
    return f"Saved: {term} → {translation} ({target_language})"


def add_to_translation_memory(text, source_language, target_language, translation):
    text, translation = (text or "").strip(), (translation or "").strip()
    if not text or not translation:
        return "Nothing to save. Translate some text first."
    if source_language == target_language:
        return "Source and target language are the same."
    if not RAG_READY:
        return "RAG is not available."
    doc_id = f"tm::{source_language}::{target_language}::{' '.join(text.casefold().split())}"
    tm_store.add_documents(
        [Document(page_content=text, metadata={"kind": "translation_memory",
                                               "source_language": source_language,
                                               "target_language": target_language,
                                               "translation": translation})],
        ids=[doc_id]
    )
    _append_csv(TM_CSV, TM_FIELDS, {"source_language": source_language, "source_text": text,
                                    "target_language": target_language, "translation": translation})
    return "Saved to translation memory."


def backup_vector_db_to_drive():
    """Only needed if the local fallback DB was used."""
    if DB_PATH == DB_PRIMARY:
        return "Vector DB already lives on Google Drive: " + DB_PRIMARY
    shutil.copytree(DB_PATH, DB_PRIMARY + "_backup", dirs_exist_ok=True)
    return "Backup copied to " + DB_PRIMARY + "_backup"


print("Database update helpers ready.")


Database update helpers ready.


In [17]:
# VERBA 2.0 — Step 8: Baseline vs RAG comparison and retrieval check (for the project report)

demo_sentence = "Welcome to Palghar."
demo_languages = ["Hindi", "Marathi", "Gujarati", "Bengali", "Tamil", "Telugu"]

print("VERBA RAG Comparison")
print("-" * 50)
print("Input:", demo_sentence)

for language in demo_languages:
    baseline = translate(demo_sentence, "English", language)
    rag_info = translate_with_rag(demo_sentence, "English", language)

    print(f"\n{language}")
    print("  Baseline NLLB :", baseline)
    print("  With RAG      :", rag_info["translation"])
    print("  Method        :", rag_info["method"])

print("\n" + "-" * 50)
print("Top-3 glossary retrieval check")

for query in ["Welcome to Palghar.", "Travel from Mumbai to Maharashtra", "Good morning."]:
    print("\nQuery:", query)
    for term, translated, similarity in retrieve_top_glossary(query, "English", "Hindi"):
        print(f"  {term} -> {translated}  (cosine {similarity})")


VERBA RAG Comparison
--------------------------------------------------
Input: Welcome to Palghar.

Hindi
  Baseline NLLB : Palghar में आपका स्वागत है.
  With RAG      : पालघर में आपका स्वागत है।
  Method        : NLLB + Glossary RAG

Marathi
  Baseline NLLB : पालघरमध्ये आपले स्वागत आहे.
  With RAG      : पालघर मध्ये आपले स्वागत आहे.
  Method        : NLLB + Glossary RAG

Gujarati
  Baseline NLLB : પાલ્ઘરમાં આપનું સ્વાગત છે.
  With RAG      : પાલઘર પર આપનું સ્વાગત છે.
  Method        : NLLB + Glossary RAG

Bengali
  Baseline NLLB : পলগারে স্বাগতম।
  With RAG      : স্বাগতম পালঘর.
  Method        : NLLB + Glossary RAG

Tamil
  Baseline NLLB : Palghar வரவேற்கிறோம்.
  With RAG      : வரவேற்கிறோம் பால்கர்.
  Method        : NLLB + Glossary RAG

Telugu
  Baseline NLLB : పల్ఘర్ కు స్వాగతం.
  With RAG      : పాల్ఘర్ కు స్వాగతం.
  Method        : NLLB + Glossary RAG

--------------------------------------------------
Top-3 glossary retrieval check

Query: Welcome to Palghar.
  Palghar -> पालघर

In [18]:
# ============================================
# VERBA 2.0 — TRANSLATOR WITH RAG + VECTOR DATABASE
# (same interface as VERBA 2.0, plus RAG panel)
# ============================================

import gradio as gr

# Close the earlier VERBA 2.0 app (if it is still running) before launching this one
try:
    demo.close()
except Exception:
    pass


def run_translation_rag(text, source_language, target_language, use_rag):

    if not text or not text.strip():
        return "Please enter text to translate.", ""

    try:
        info = translate_with_rag(text, source_language, target_language, use_rag)
        return info["translation"], format_rag_report(info)

    except Exception as error:
        return f"Translation Error: {str(error)}", ""


def swap_languages(source, target):
    return target, source


def clear_fields():
    return "", "", ""


def save_glossary_term(term, target_language, translation):
    try:
        return add_glossary_term(term, target_language, translation)
    except Exception as error:
        return f"Could not save: {error}"


def save_memory(text, source_language, target_language, translation):
    try:
        return add_to_translation_memory(text, source_language, target_language, translation)
    except Exception as error:
        return f"Could not save: {error}"


custom_css = """
.gradio-container {
    background: linear-gradient(135deg, #10172e, #20204a, #301b52);
}

/* Header */
#main-title {
    text-align: center;
    padding: 30px;
    border-radius: 18px;
    background: linear-gradient(90deg, #292e63, #663399);
}

#main-title h1 {
    font-size: 45px;
    color: #ffffff !important;
}

#main-title p {
    color: #e1dcff !important;
}

/* Make all Markdown text visible */
.prose,
.prose p,
.prose h1,
.prose h2,
.prose h3,
.prose h4,
.prose li,
.prose strong,
.prose span {
    color: #ffffff !important;
}

/* Input labels */
label {
    color: #ffffff !important;
}

/* Buttons */
.gr-button-primary {
    background: linear-gradient(90deg, #6255e7, #9b59d0) !important;
    color: white !important;
    border: none !important;
}

/* Footer text */
footer {
    color: #ffffff !important;
}
"""


with gr.Blocks(title="VERBA - AI Translator") as rag_demo:

    gr.HTML("""
    <div id="main-title">
        <h1>🌐 VERBA</h1>
        <p>Connecting Languages, Connecting People</p>
        <p>AI-Powered Multilingual Translation Platform · RAG + Vector Database</p>
    </div>
    """)

    gr.Markdown("## ✨ Translate your thoughts into another language")

    with gr.Row():

        source_language = gr.Dropdown(
            choices=list(LANGUAGES.keys()),
            value="English",
            label="🌍 Translate From"
        )

        swap_button = gr.Button("⇄ Swap Languages")

        target_language = gr.Dropdown(
            choices=list(LANGUAGES.keys()),
            value="Marathi",
            label="🎯 Translate To"
        )

    with gr.Row():

        input_text = gr.Textbox(
            label="📝 Enter Your Text",
            placeholder="Type something here...",
            lines=8
        )

        output_text = gr.Textbox(
            label="🌐 Translation Result",
            placeholder="Your translation appears here...",
            lines=8,
            interactive=False
        )

    use_rag = gr.Checkbox(
        label="🔎 Use RAG (vector database: glossary + translation memory)",
        value=True
    )

    with gr.Row():

        translate_button = gr.Button("✨ Translate Now", variant="primary")

        clear_button = gr.Button("🗑 Clear")

    rag_report = gr.Markdown(value="")

    with gr.Row():

        save_memory_button = gr.Button("✅ Save this translation to memory")

        memory_status = gr.Textbox(label="Memory status", interactive=False)

    with gr.Accordion("📚 Add a verified glossary term", open=False):

        glossary_term = gr.Textbox(label="Term (English)", placeholder="e.g. Palghar")

        glossary_translation = gr.Textbox(
            label="Approved translation in the 'Translate To' language"
        )

        glossary_button = gr.Button("Add to glossary")

        glossary_status = gr.Textbox(label="Glossary status", interactive=False)

    gr.Markdown("## 💡 Try These Examples")

    gr.Examples(
        examples=[
            ["Hello, how are you?", "English", "Hindi"],
            ["Welcome to Palghar.", "English", "Marathi"],
            ["Education is the key to success.", "English", "Gujarati"],
            ["Thank you for your help.", "English", "Spanish"],
            ["Good morning.", "English", "Japanese"],
            ["Technology connects the world.", "English", "French"],
            ["Knowledge is power.", "English", "German"],
            ["Welcome to Palghar.", "English", "Tamil"]
        ],
        inputs=[input_text, source_language, target_language]
    )

    gr.Markdown("""
    ---

    ## 🚀 About VERBA

    VERBA uses Meta AI's pretrained
    **NLLB-200 Distilled 600M** translation model.

    ### Features

    - 22 language options
    - AI-powered text translation
    - Language swapping
    - Easy-to-use interface
    - Multiple language support
    - RAG with a ChromaDB vector database (LangChain)
    - Glossary protection for local names and terms
    - Translation memory for verified sentences

    ---

    ⚠️ Academic translation prototype.
    Please verify translations before important use.

    <center>
    Built with Python, PyTorch, LangChain, ChromaDB and Gradio
    | VERBA 3.0
    </center>
    """)

    translate_button.click(
        fn=run_translation_rag,
        inputs=[input_text, source_language, target_language, use_rag],
        outputs=[output_text, rag_report]
    )

    swap_button.click(
        fn=swap_languages,
        inputs=[source_language, target_language],
        outputs=[source_language, target_language]
    )

    clear_button.click(
        fn=clear_fields,
        inputs=[],
        outputs=[input_text, output_text, rag_report]
    )

    save_memory_button.click(
        fn=save_memory,
        inputs=[input_text, source_language, target_language, output_text],
        outputs=memory_status
    )

    glossary_button.click(
        fn=save_glossary_term,
        inputs=[glossary_term, target_language, glossary_translation],
        outputs=glossary_status
    )


rag_demo.launch(
    share=True,
    theme=gr.themes.Soft(),
    css=custom_css
)


Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://49d6f39ec83e63d004.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
